# Predictive modeling

The reproducible training implementation is in `src/modeling.py`. This notebook validates and explores artifacts produced by the shared modeling code.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "deployment").exists():
    root = root.parent
snapshot = root / "deployment/snapshot"
profile = json.loads((snapshot / "data_report.json").read_text())
model = json.loads((snapshot / "model.json").read_text())
def sql(name):
    return pd.read_csv(snapshot / "analytics" / f"{name}.csv")


In [2]:
import duckdb
from src.modeling import FEATURES, TRAIN_END, CAL_END, TEST_START, temporal_frames

with duckdb.connect(str(root/"data/claims.duckdb"),read_only=True) as con:
    features = con.execute((root/"sql/features.sql").read_text()).df()
truth = pd.read_parquet(root/"data/claims.truth.parquet")
splits = temporal_frames(features,truth)
for name, cutoff in [("train",TRAIN_END),("calibration",CAL_END),("validation",TEST_START)]:
    assert splits[name].mature_at.max() < cutoff
print("Predictor allowlist:", FEATURES)
display(pd.Series({name:len(part) for name,part in splits.items()},name="claims").to_frame())


Predictor allowlist: ['severity', 'complexity', 'estimate', 'reserve', 'report_delay', 'workload', 'documented', 'early_events', 'liability', 'vendor_days', 'vendor_supp', 'coverage']


,claims
train,52548
calibration,6932
validation,12840
test,29135


In [3]:
display(pd.DataFrame(model["validation"])[["model","selection_ap","ap","brier","precision_at_k"]])
display(pd.DataFrame(model["amount_validation"]))
print(model["selected"], "|", model["amount_model"], "|", model["threshold"])
display(pd.Series(model["test"],name="test").to_frame())


,model,selection_ap,ap,brier,precision_at_k
0,Operational rules,0.094023,0.092777,0.066259,0.105140
1,Logistic regression,0.121798,0.113836,0.065625,0.144081
2,Random forest,0.113766,0.106819,0.067458,0.141745
3,LightGBM depth 2,0.117273,0.110862,0.065648,0.147975
4,LightGBM depth 4,0.113469,0.106899,0.067679,0.128505


,model,mae,rmse
0,Severity mean,264.622556,388.134199
1,Random forest,191.587486,306.139547
2,LightGBM,191.269607,302.684399


Logistic regression | LightGBM | 0.10440457021369567


,test
ap,0.102558
roc_auc,0.620807
brier,0.064903
precision,0.137828
recall,0.135148
confusion,"[[25339, 1739], [1779, 278]]"
precision_at_k,0.132853
recall_at_k,0.188138
dollars_at_k,226274.63
ap_month_block_ci,"[0.08926792109850124, 0.11673003382325108]"


Risk selection uses unbinned validation AP with a 0.002 preference for a simpler candidate. Isotonic calibration uses a separate chronological period. Amount selection minimizes validation RMSE to target a conditional mean; MAE and empirical interval coverage offer complementary diagnostics.